# Imports

In [10]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from rank_bm25 import BM25Okapi
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder
import openai
import numpy as np
import re
import os
import json

# 1. Get Embedding Model

In [2]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)


C:\Users\Reddy\AppData\Local\Temp\ipykernel_2728\63606858.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 2. Get DataBase¶

In [3]:
# 1. Initialize DB 
db = Chroma(
    collection_name='subtitle_vector_database',
    embedding_function=embedding_model,
    persist_directory='./new_chroma_db'
)



# 3. HybridRetriever

**Purpose:**
  Combines BM25 (keyword matching and dense embeddings (semantic similarity) to retrieve relevant documetns balancing lexical and semantic signals improves robustness across different query types.

**How it works**:
- BM25 scores → capture exact term overlap
- Embedding similarity → capture semantic meaning
- Final score = weighted combination using alpha
  

In [4]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

In [5]:

class HybridRetriever:
    def __init__(self, data, embedding_model):
        self.data = data
        self.doc_embeddings = data['embeddings']
        self.embedding_model = embedding_model
        self.bm25 = BM25Okapi([tokenize(doc) for doc in data['documents']])
        
    def search(self, query,k=4, alpha = 0.5):
        
        #keyword scores
        bm25_scores = np.array(self.bm25.get_scores(tokenize(query)))
        bm25_scores = np.log1p(bm25_scores)
        if bm25_scores.max() > 0:
            bm25_scores = bm25_scores / bm25_scores.max()
            
        #semantic scores
        embedded_query = self.embedding_model.embed_query(query)
        embedded_query = embedded_query / (np.linalg.norm(embedded_query) + 1e-8)

        doc_vecs = self.doc_embeddings / (np.linalg.norm(self.doc_embeddings, axis=1, keepdims=True) + 1e-8)


        sem_scores = np.dot(doc_vecs, embedded_query)
        
        combined_score = alpha*sem_scores + (1-alpha) * bm25_scores
        top_k_indices = np.argsort(combined_score)[::-1][:k]
        
        results = []
        for i in top_k_indices:
            results.append({
                'id':i,
                'text': self.data['documents'][i],
                'metadata': self.data['metadatas'][i]
            })
        return results

# 4. RerankedRetriever
**Purpose**:
Refines retrieved results by re-evaluating query-document pairs using a cross-encoder impoving precision by correcting ranking errors from the initial retrieval stage.

**How it works:**

- Takes top candidate_k results from hybrid retriever
- Scores each (query, document) pair jointly
- Reorders results based on relevance scores


In [6]:
class RerankedRetriever:
    def __init__(self, hybrid_retriever):
        self.hybrid = hybrid_retriever
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        
    def search(self, query, k=5, alpha =0.6, candidate_k = 20):

        #large pool of candidates from hybrid retriever
        candidates = self.hybrid.search(query, k= candidate_k, alpha = alpha)
        
        #prepare query-doc pair for rerenker to get relevant score
        pairs = [(query, doc['text']) for doc in candidates]
        
        scores = self.reranker.predict(pairs)
        
        reranked = sorted(zip(candidates, scores), key = lambda x: x[1], reverse = True)
        
        return [
                    {   "id": int(doc["id"]),
                        "text": doc["text"],
                        "metadata": doc["metadata"],
                        "score": float(score)
                    }
                    for doc, score in reranked[:k]
                ]

# 5. LLM Call

**Purpose:**
Generate a grounded answer using retrieved context.The LLM does not retrieve information—it only reasons over the provided context. Final answer quality depends heavily on retrieval quality and prompt constraints.

**How it works:**

- Retrieves top documents from retriever
- Formats them into a clean context string
- Uses a constrained prompt to force context-based answering
- Calls the LLM with temperature=0 for consistent outputs



In [7]:
#get api key
f= open('openai_api_key.txt')
OPENAI_API_KEY = f.read()

In [8]:
def generate_response(
    query,
    retriever,
    llm_client,
    model="gpt-5.2",
    k=3,
    temperature=0
):
    # 1. Retrieve
    search_results = retriever.search(query=query, k=10)

    # 2. Build context (LLM-readable, not raw dicts)
    context_blocks = []
    for res in search_results:
        meta = res.get("metadata", {})
        content = res.get("text", "")
        
        # compact + consistent metadata
        source_info = f"[S{meta.get('season')}E{meta.get('episode')}]"
        context_blocks.append(f"{source_info}\n{content}")
    
    context_str = "\n\n".join(context_blocks)

    # 3. Prompt (strict for evaluation)
    prompt = f"""
        You are a question answering system for the FRIENDS TV series.
        
        STRICT RULES:
        - Answer ONLY using the provided context.
        - Do NOT use outside knowledge.
        - Prefer answers directly supported by context.
        - If the answer is not explicitly stated but can be strongly inferred from the context, provide the most likely answer.
        - When making an inference, briefly justify your answer using the context.
        - Do not rely purely on general knowledge.
        - If the question refers to an episode not present in the context metadata, respond that it is not available in the database.Do not talk anything about context
        - If there is insufficient information even to infer, say "I don't know."
        - Keep the answer short (1-2 sentences max).
        - If the question asks for a summary or overview, use all relevant context available and provide a best-effort summary.
        - If only partial data is available, mention that the summary is based on available episodes from the context.
           
        Context:
            ```{context_str}```
        Question: {query}
        Answer:
    """
    # 4. LLM call
    response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )

    answer = response.choices[0].message.content.strip()

      # 5. Return everything needed for evaluation/debugging
    return {
        "query": query,
        "retrieved_chunks": search_results,
        "context": context_str,
        "answer": answer
    }
    

# 6. Evaluator

**Purpose:**  
Compute average Precision@k and Recall@k over a dataset to assess retriever performance.

**How it works:**  
- Iterates over each query in the dataset  
- Retrieves top-k documents using the retriever  
- Compares retrieved document IDs with ground truth  
- Computes precision and recall per query  
- Returns the average precision and recall across all queries  

In [29]:
def evaluate(retriever, dataset, k, **search_kwargs):
    all_precision = []
    all_recall = []

    for item in dataset:
        query = item['query']
        actual = set(item['relevant_docs'])

        results = retriever.search(query=query, k=k, **search_kwargs)
        predicted = set(doc['id'] for doc in results)

        tp = actual & predicted

        precision = len(tp) / len(predicted) if predicted else 0
        recall = len(tp) / len(actual) if actual else 0

        all_precision.append(precision)
        all_recall.append(recall)

    return {
        "precision": sum(all_precision) / len(all_precision),
        "recall": sum(all_recall) / len(all_recall)
    }

In [30]:
#get chunks from vector database along with embeddings and metadata
data = db.get(include = ['embeddings','documents','metadatas'])

#Initialize hybrid retriever
my_hybrid_retriever = HybridRetriever(data = data,embedding_model=embedding_model )

# Initialize reranker as a whole retrieer(hybrid search +reraking0)
reranked_retriever= RerankedRetriever(my_hybrid_retriever)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Test Golden Set

In [11]:
with open("friends_golden_set.json", "r", encoding="utf-8") as f:
    golden_eval_set = json.load(f)

### First Experiment: Initial Baseline

**Setup:**  
Used initial parameter values (alpha=0.7, candidate_k=50, k=7) to establish a baseline.

In [31]:
results = evaluate(candidate_k= 50, k=7, retriever=reranked_retriever, dataset = golden_eval_set)

print(f"alpha = 0.7, candidate_k = 50, top_k = 7 → Precision@7: {results['precision']:.3f}, Recall@7: {results['recall']:.3f}")

alpha = 0.7, candidate_k = 50, top_k = 7 → Precision@7: 0.117, Recall@7: 0.640


**Result:**  
Precision@7: 0.112  
Recall@7: 0.640  

**Insight:**  
Recall is relatively high, but precision is low, indicating that while many relevant documents are retrieved, a significant number of irrelevant results are also included.

**Note:**  
This serves as a baseline for further tuning.

### Experiment: Alpha Tuning (Hybrid Retriever)

**Setup:**  
Evaluated hybrid retriever performance by varying alpha ∈ {0.3, 0.5, 0.7, 0.9} with k=50.

In [35]:
for alpha in [0.3, 0.5, 0.7, 0.9]:
    res = evaluate(my_hybrid_retriever, golden_eval_set, k=50, alpha=alpha)
    print(f"alpha={alpha} → Precision@50: {res['precision']:.3f}, Recall@50: {res['recall']:.3f}")
    print("-"*60)

alpha=0.3 → Precision@50: 0.022, Recall@50: 0.880
------------------------------------------------------------
alpha=0.5 → Precision@50: 0.024, Recall@50: 0.940
------------------------------------------------------------
alpha=0.7 → Precision@50: 0.024, Recall@50: 0.940
------------------------------------------------------------
alpha=0.9 → Precision@50: 0.023, Recall@50: 0.900
------------------------------------------------------------


**Results:**

| Alpha | Precision@50 | Recall@50 |
|------:|-------------:|----------:|
| 0.3   | 0.022        | 0.880     |
| 0.5   | 0.024        | 0.940     |
| 0.7   | 0.024        | 0.940     |
| 0.9   | 0.023        | 0.900     |

**Observation:**  
Performance improves from alpha=0.3 to 0.5/0.7, then slightly drops at 0.9.

**Insight:**  
A balanced combination of lexical and semantic signals (alpha ≈ 0.5–0.7) yields better recall, while too much reliance on semantic similarity (alpha=0.9) slightly degrades performance.

**Note:**  
Differences in precision are minimal, suggesting recall is more sensitive to alpha in this setup.

### Experiment: Candidate_k Tuning (Hybrid Retriever, alpha = 0.6)

**Setup:**  
Evaluated hybrid retriever by varying candidate_k with alpha fixed at 0.6.

In [36]:
candidate_ks = [5,10,15,20,25,30,35,40,45]
for ck in candidate_ks:
    res = evaluate(my_hybrid_retriever, golden_eval_set, k=ck, alpha=0.6)
    print(f"Alpha = 0.6, candidate_k={ck}, Precision@{ck}={res['precision']:.3f}, Recall@{ck}={res['recall']:.3f}")
    print("-"*60)

Alpha = 0.6, candidate_k=5, Precision@5=0.164, Recall@5=0.630
------------------------------------------------------------
Alpha = 0.6, candidate_k=10, Precision@10=0.100, Recall@10=0.770
------------------------------------------------------------
Alpha = 0.6, candidate_k=15, Precision@15=0.073, Recall@15=0.860
------------------------------------------------------------
Alpha = 0.6, candidate_k=20, Precision@20=0.055, Recall@20=0.860
------------------------------------------------------------
Alpha = 0.6, candidate_k=25, Precision@25=0.046, Recall@25=0.900
------------------------------------------------------------
Alpha = 0.6, candidate_k=30, Precision@30=0.039, Recall@30=0.920
------------------------------------------------------------
Alpha = 0.6, candidate_k=35, Precision@35=0.034, Recall@35=0.940
------------------------------------------------------------
Alpha = 0.6, candidate_k=40, Precision@40=0.030, Recall@40=0.940
--------------------------------------------------------

**Results:**

| candidate_k | Precision@k | Recall@k |
|------------:|------------:|---------:|
| 5           | 0.164       | 0.630    |
| 10          | 0.100       | 0.770    |
| 15          | 0.073       | 0.860    |
| 20          | 0.055       | 0.860    |
| 25          | 0.046       | 0.900    |
| 30          | 0.039       | 0.920    |
| 35          | 0.034       | 0.940    |
| 40          | 0.029       | 0.940    |
| 45          | 0.026       | 0.940    |

**Observation:**  
Recall increases with candidate_k, while precision steadily decreases.

**Insight:**  
Larger candidate pools improve coverage but introduce more irrelevant documents, leading to lower precision.

**Note:**  
Recall plateaus after ~35, suggesting diminishing returns beyond this point.

### Experiment: k Tuning (Reranked Retriever, alpha = 0.6, candidate_k = 35)

**Setup:**  
Evaluated reranked retriever by varying k with alpha=0.6 and candidate_k=35.


In [48]:
k_values = [1, 3, 5, 10, 15, 20, 25] 

for k in k_values:
    res = evaluate(retriever, golden_eval_set, k=k, alpha=0.6, candidate_k=35)
    print(f"Alpha = 0.6, candidate_k=35, top_k = {k}, Precision@{k}={res['precision']:.3f}, Recall@{k}={res['recall']:.3f}")
    print("-"*60)

Alpha = 0.6, candidate_k=35, top_k = 1, Precision@1=0.360, Recall@1=0.310
------------------------------------------------------------
Alpha = 0.6, candidate_k=35, top_k = 3, Precision@3=0.167, Recall@3=0.410
------------------------------------------------------------
Alpha = 0.6, candidate_k=35, top_k = 5, Precision@5=0.152, Recall@5=0.600
------------------------------------------------------------
Alpha = 0.6, candidate_k=35, top_k = 10, Precision@10=0.092, Recall@10=0.710
------------------------------------------------------------
Alpha = 0.6, candidate_k=35, top_k = 15, Precision@15=0.072, Recall@15=0.840
------------------------------------------------------------
Alpha = 0.6, candidate_k=35, top_k = 20, Precision@20=0.056, Recall@20=0.880
------------------------------------------------------------
Alpha = 0.6, candidate_k=35, top_k = 25, Precision@25=0.046, Recall@25=0.920
------------------------------------------------------------


**Results:**

| k  | Precision@k | Recall@k |
|---:|------------:|---------:|
| 1  | 0.360       | 0.310    |
| 3  | 0.167       | 0.410    |
| 5  | 0.152       | 0.600    |
| 10 | 0.092       | 0.710    |
| 15 | 0.072       | 0.840    |
| 20 | 0.056       | 0.880    |
| 25 | 0.046       | 0.920    |

**Observation:**  
Precision decreases while recall increases as k grows.

**Insight:**  
Smaller k gives more precise top results, while larger k improves coverage but introduces noise.

**Note:**  
Recall continues improving up to k=25, suggesting larger result sets capture more relevant documents.

# Test Adversarial Set

In [38]:
with open("friends_adversarial_set.json", 'r', encoding = 'utf-8') as f:
    adversarial_eval_set = json.load(f)

### First Experiment: Initial Baseline

**Setup:**  
Used initial parameter values (alpha=0.7, candidate_k=50, k=7) to establish a baseline.

In [40]:
results = evaluate(candidate_k= 50, k=7, retriever=reranked_retriever, dataset = adversarial_eval_set)

print(f"alpha = 0.7, candidate_k = 50, top_k = 7 → Precision@7: {results['precision']:.3f}, Recall@7: {results['recall']:.3f}")

alpha = 0.7, candidate_k = 50, top_k = 7 → Precision@7: 0.139, Recall@7: 0.391


**Result:**  
Precision@7: 0.139
Recall@7: 0.39

**Observation:**  
Recall and precision are significantly lower compared to the golden set for the same parameters.

**Insight:**  
The retriever struggles more on adversarial queries, likely due to weaker lexical overlap and less direct semantic alignment, leading to missed relevant documents and increased noise.

**Note:**  
This serves as a baseline for further tuning.

### Experiment: Alpha Tuning (Hybrid Retriever)

**Setup:**  
Evaluated hybrid retriever performance by varying alpha ∈ {0.3, 0.5, 0.7, 0.9} with k=50.

In [41]:
for alpha in [0.3, 0.5, 0.7, 0.9]:
    res = evaluate(my_hybrid_retriever, adversarial_eval_set, k=50, alpha=alpha)
    print(f"alpha={alpha} → Precision@50: {res['precision']:.3f}, Recall@50: {res['recall']:.3f}")
    print("-"*60)

alpha=0.3 → Precision@50: 0.033, Recall@50: 0.633
------------------------------------------------------------
alpha=0.5 → Precision@50: 0.035, Recall@50: 0.707
------------------------------------------------------------
alpha=0.7 → Precision@50: 0.034, Recall@50: 0.694
------------------------------------------------------------
alpha=0.9 → Precision@50: 0.033, Recall@50: 0.683
------------------------------------------------------------


**Observation:**  
Performance improves from alpha=0.3 to 0.5, then slightly drops at 0.7.

**Insight:**  
A balanced combination of lexical and semantic signals (alpha ≈ 0.5) yields better recall, while too much reliance on semantic similarity (alpha>0.7) slightly degrades performance.

**Note:**  
Compared to the golden set, overall recall is lower, indicating greater sensitivity to query variation.

### Experiment: Candidate_k Tuning (Hybrid Retriever, alpha = 0.5)

**Setup:**  
Evaluated hybrid retriever by varying candidate_k with alpha fixed at 0.5.

In [46]:
candidate_ks = [5,10,15,20,25,30,35,40,45,50]
for ck in candidate_ks:
    res = evaluate(my_hybrid_retriever, adversarial_eval_set, k=ck, alpha=0.5)
    print(f"Alpha = 0.5, candidate_k={ck}, Precision@{ck}={res['precision']:.3f}, Recall@{ck}={res['recall']:.3f}")
    print("-"*60)

Alpha = 0.5, candidate_k=5, Precision@5=0.116, Recall@5=0.238
------------------------------------------------------------
Alpha = 0.5, candidate_k=10, Precision@10=0.105, Recall@10=0.397
------------------------------------------------------------
Alpha = 0.5, candidate_k=15, Precision@15=0.077, Recall@15=0.451
------------------------------------------------------------
Alpha = 0.5, candidate_k=20, Precision@20=0.064, Recall@20=0.493
------------------------------------------------------------
Alpha = 0.5, candidate_k=25, Precision@25=0.059, Recall@25=0.572
------------------------------------------------------------
Alpha = 0.5, candidate_k=30, Precision@30=0.051, Recall@30=0.590
------------------------------------------------------------
Alpha = 0.5, candidate_k=35, Precision@35=0.045, Recall@35=0.607
------------------------------------------------------------
Alpha = 0.5, candidate_k=40, Precision@40=0.041, Recall@40=0.629
--------------------------------------------------------

**Results:**

| candidate_k | Precision@k | Recall@k |
|------------:|------------:|---------:|
| 5           | 0.116       | 0.238    |
| 10          | 0.105       | 0.397    |
| 15          | 0.077       | 0.451    |
| 20          | 0.064       | 0.493    |
| 25          | 0.059       | 0.572    |
| 30          | 0.051       | 0.590    |
| 35          | 0.045       | 0.607    |
| 40          | 0.041       | 0.629    |
| 45          | 0.038       | 0.695    |
| 50          | 0.035       | 0.707    |

**Observation:**  
Recall increases with candidate_k, while precision steadily decreases.

**Insight:**  
Relevant documents are not consistently ranked at top positions and require larger candidate pools to be retrieved, indicating weaker initial retrieval performance.

**Note:**  
Compared to the golden set, higher candidate_k is required to achieve similar recall, suggesting greater retrieval difficulty rather than just parameter sensitivity.

### Experiment: k Tuning (Reranked Retriever, alpha = 0.5, candidate_k = 50)

**Setup:**  
Evaluated reranked retriever by varying k with alpha=0.5 and candidate_k=50.


In [50]:
k_values = [1, 3, 5, 10, 15, 20, 25] 

for k in k_values:
    res = evaluate(retriever, adversarial_eval_set, k=k, alpha=0.5, candidate_k=50)
    print(f"Alpha = 0.5, candidate_k=50, top_k = {k}, Precision@{k}={res['precision']:.3f}, Recall@{k}={res['recall']:.3f}")
    print("-"*60)

Alpha = 0.5, candidate_k=50, top_k = 1, Precision@1=0.342, Recall@1=0.159
------------------------------------------------------------
Alpha = 0.5, candidate_k=50, top_k = 3, Precision@3=0.184, Recall@3=0.236
------------------------------------------------------------
Alpha = 0.5, candidate_k=50, top_k = 5, Precision@5=0.137, Recall@5=0.271
------------------------------------------------------------
Alpha = 0.5, candidate_k=50, top_k = 10, Precision@10=0.118, Recall@10=0.471
------------------------------------------------------------
Alpha = 0.5, candidate_k=50, top_k = 15, Precision@15=0.089, Recall@15=0.526
------------------------------------------------------------
Alpha = 0.5, candidate_k=50, top_k = 20, Precision@20=0.074, Recall@20=0.571
------------------------------------------------------------
Alpha = 0.5, candidate_k=50, top_k = 25, Precision@25=0.061, Recall@25=0.593
------------------------------------------------------------


**Results:**

| k  | Precision@k | Recall@k |
|---:|------------:|---------:|
| 1  | 0.342       | 0.159    |
| 3  | 0.184       | 0.236    |
| 5  | 0.137       | 0.271    |
| 10 | 0.118       | 0.471    |
| 15 | 0.089       | 0.526    |
| 20 | 0.074       | 0.571    |
| 25 | 0.061       | 0.593    |

**Observation:**  
Recall increases with k, while precision decreases.

**Insight:**  
With tuned parameters, recall improves significantly (~59%), but at the cost of lower precision (~6%), indicating that more relevant documents are retrieved but with increased noise.

**Note:**  
This reflects the inherent trade-off between precision and recall, especially pronounced in adversarial queries.

# Why did Precision decrease?

The observed drop in precision can be attributed to three main factors:

**1. Ambiguity in conversational data**

Conversational datasets inherently contain ambiguity, implicit context, and fragmented information. As a result, many retrieved chunks are only partially relevant to the query. However, if the evaluation setup requires exact matches with predefined ground truth, these partially relevant chunks are counted as false positives, thereby lowering precision.

**2. Effect of increasing `candidate_k`**

Increasing `candidate_k` improves recall by expanding the search space and retrieving more potentially relevant chunks. However, this also introduces additional noise. Since precision depends on the proportion of relevant items among retrieved items, a larger candidate pool increases the denominator, leading to a drop in precision even if recall improves.



# Overall Observation

1. The golden set offers greater scope for improving recall because the queries are more direct and closely aligned with the underlying documents. In contrast, adversarial queries are more challenging due to a combination of factors, including dispersed relevance, paraphrasing, reduced lexical overlap, and potential mismatch with the embedding space.
2. For direct queries, it is more efficient to use smaller candidate sets and lower `top_k` values. This reduces computational and memory overhead while maintaining performance. However, this assumption may not hold for ambiguous or adversarial queries, where a larger candidate pool can help capture scattered or indirectly relevant information.
3. The reranker may reduce recall compared to the hybrid retriever, but removing it based solely on this observation would be short-sighted. The reranker achieves comparable or better precision using significantly fewer documents than the hybrid retriever (e.g., compared to retrieving 50 documents). This makes the trade-off acceptable, especially in systems where efficiency and response quality are critical.
4. As the number of documents included in the prompt increases, large language models become more susceptible to the `needle in the haystack` problem, where relevant information is harder to identify among noise. This degradation directly impacts response quality.
In worst-case scenarios, the model may generate responses based on irrelevant or misleading context. This further justifies the inclusion of a reranker, as it helps filter noise and prioritize the most relevant information before passing it to the model.
5. Conversational data often contains events, dialogues, character references, and implicit context. Due to this complexity, relying solely on either semantic or keyword-based retrieval is insufficient. Assigning balanced weight to both approaches improves robustness by capturing both exact matches and contextual meaning.